Imports and Setup

In [ ]:
""" 
- 02_api_call.ipynb — Full Data Extraction Loop
- Fetches search results from Rainforest API across 4 categories and 15 pages each, totaling 60 requests (estimated).
- Raw responses are saved as JSON files for downstream processing. """

import requests
import json
import time
import os
from dotenv import load_dotenv

# Load API credentials from .env file
load_dotenv()
API_KEY = os.getenv("RAINFOREST_API_KEY")
API_KEY_2 = os.getenv("RAINFOREST_API_KEY_2")

# API configuration
BASE_URL = "https://api.rainforestapi.com/request"
OUTPUT_DIR = "../data/api_raw"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [14]:
# Search categories: key used for filenames, value sent to the API
CATEGORIES = {
    "electronics": "best electronics",
    "home_kitchen": "best home kitchen products",
    "beauty_personal_care": "best beauty personal care",
    "sports_outdoors": "best sports outdoors"
}

PAGES = range(1, 16)  # 1 to 15

Function

In [ ]:
def fetch_search_page(category_key: str, search_term: str, page: int) -> None:
    """Fetch one search results page and save raw JSON to disk."""
    filepath = os.path.join(OUTPUT_DIR, f"{category_key}_page_{page}.json")

    # if the file already exists then return for not wasting requests
    if os.path.exists(filepath):
        print(f"[SKIP] {category_key} p{page}")
        return
    
    params = {
        "api_key": API_KEY_2,
        "type": "search",
        "amazon_domain": "amazon.com",
        "search_term": search_term,
        "page": page,
    }

    response = requests.get(BASE_URL, params=params)
    # if the request is not 200 then print the error and status
    if response.status_code != 200:
        print(f"[ERROR] {category_key} p{page} — status {response.status_code}")
        return
    # create json files if the request was succesful
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(response.json(), f, ensure_ascii=False, indent=2)

    print(f"[OK] {category_key} p{page}")
    time.sleep(0.5) # wait half a second for the API call to avoid overloading it



Starting category: ELECTRONICS
[OK] electronics p1
[OK] electronics p2
[OK] electronics p3
[OK] electronics p4
[OK] electronics p5
[OK] electronics p6
[OK] electronics p7
[OK] electronics p8
[OK] electronics p9
[OK] electronics p10
[OK] electronics p11
[OK] electronics p12
[OK] electronics p13
[OK] electronics p14
[OK] electronics p15

Starting category: HOME_KITCHEN
[OK] home_kitchen p1
[OK] home_kitchen p2
[OK] home_kitchen p3
[OK] home_kitchen p4
[OK] home_kitchen p5
[OK] home_kitchen p6
[OK] home_kitchen p7
[OK] home_kitchen p8
[OK] home_kitchen p9
[OK] home_kitchen p10
[OK] home_kitchen p11
[OK] home_kitchen p12
[OK] home_kitchen p13
[OK] home_kitchen p14
[OK] home_kitchen p15

Starting category: BEAUTY_PERSONAL_CARE
[OK] beauty_personal_care p1
[OK] beauty_personal_care p2
[OK] beauty_personal_care p3
[OK] beauty_personal_care p4
[OK] beauty_personal_care p5
[OK] beauty_personal_care p6
[OK] beauty_personal_care p7
[OK] beauty_personal_care p8
[OK] beauty_personal_care p9
[OK] b

Execution

In [16]:
# 4 categories x 15 pages = 60 requests
for category_key, search_term in CATEGORIES.items():
    print(f"\n--- {category_key} ---")
    for page in PAGES:
        fetch_search_page(category_key, search_term, page)




--- electronics ---
[SKIP] electronics p1
[SKIP] electronics p2
[SKIP] electronics p3
[SKIP] electronics p4
[SKIP] electronics p5
[SKIP] electronics p6
[SKIP] electronics p7
[SKIP] electronics p8
[SKIP] electronics p9
[SKIP] electronics p10
[SKIP] electronics p11
[SKIP] electronics p12
[SKIP] electronics p13
[SKIP] electronics p14
[SKIP] electronics p15

--- home_kitchen ---
[SKIP] home_kitchen p1
[SKIP] home_kitchen p2
[SKIP] home_kitchen p3
[SKIP] home_kitchen p4
[SKIP] home_kitchen p5
[SKIP] home_kitchen p6
[SKIP] home_kitchen p7
[SKIP] home_kitchen p8
[SKIP] home_kitchen p9
[SKIP] home_kitchen p10
[SKIP] home_kitchen p11
[SKIP] home_kitchen p12
[SKIP] home_kitchen p13
[SKIP] home_kitchen p14
[SKIP] home_kitchen p15

--- beauty_personal_care ---
[SKIP] beauty_personal_care p1
[SKIP] beauty_personal_care p2
[SKIP] beauty_personal_care p3
[SKIP] beauty_personal_care p4
[SKIP] beauty_personal_care p5
[SKIP] beauty_personal_care p6
[SKIP] beauty_personal_care p7
[SKIP] beauty_personal_

Test API with 1 request

In [ ]:
# Test request - Sports & Outdoors page 1
params = {
    "api_key": API_KEY,
    "type": "search",
    "amazon_domain": "amazon.com",
    "search_term": "best sports outdoors",
    "page": 1
}

response = requests.get(BASE_URL, params=params)
print("Status:", response.status_code)

if response.status_code == 200:
    data = response.json()
    results = data["search_results"]
    prices_available = 0

    for item in results:    
        if item.get("price") and item.get("price", {}).get("value"):
            prices_available += 1
            
    print(f"Products retrieved: {len(results)}")

    # Inspect first product
    item = results[0]
    print("\n--- First product ---")
    print("ASIN:    ", item.get("asin"))
    print("Title:   ", item.get("title"))
    print("Rating:  ", item.get("rating"))
    print("Reviews: ", item.get("ratings_total"))
    print("Price:   ", item.get("price", {}).get("value"))
    print("Prime:   ", item.get("is_prime"))
    print("Sponsored:", item.get("sponsored"))
    print(f"Products with price: {prices_available}/{len(results)}") #show how many of the results have price (in this case 66/70)
                                                                     #after we decide what to do with the products without price
else:
    print(f"Request failed — status {response.status_code}")
    print(response.text)

Status: 200
Products retrieved: 70

--- First product ---
ASIN:     B0731VQ7VP
Title:    JP WinLook Ping Pong Paddles Sets - Portable Table Tennis Paddle Set with Ping Pong Paddles Professional Case & Ping Pong Balls - Premium Table Tennis Racket Player Set for Indoor & Outdoor Games
Rating:   4.7
Reviews:  6221
Price:    37.99
Prime:    False
Sponsored: True
Products with price: 66/70
